In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import AzureOpenAI
from embeddings import UserDocumentCollection
import os, json, re
import pandas as pd

import numpy as np
from pypdf import PdfReader
import win32com.client as win32
from time import sleep

In [16]:
load_dotenv()

True

In [3]:
folder_path = os.environ.get("FOLDER_PATH")

In [9]:
container_name="brussels-docs"
colect = UserDocumentCollection(container_name)


for p in sorted([p for p in Path(folder_path).iterdir() if p.is_file()]):
    file_path = str(p)
    with open(file_path, "rb") as file_obj:
        colect.add_file_to_blob_container(file_path, file_obj)

colect.create_user_search_index()
colect.create_data_source_connection()
colect.create_user_skillset()
colect.create_user_indexer()

2025-09-02 09:42:30.762 | INFO     | embeddings:get_container:66 - Container 'brussels-docs' already exists.
2025-09-02 09:42:30.901 | WARNING  | embeddings:add_file_to_blob_container:99 - File 'F3552803-Feedback Paper On the European Biotech Act.pdf' already exists in container 'brussels-docs'. Skipping upload.
2025-09-02 09:42:30.934 | WARNING  | embeddings:add_file_to_blob_container:99 - File 'F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf' already exists in container 'brussels-docs'. Skipping upload.
2025-09-02 09:42:30.963 | WARNING  | embeddings:add_file_to_blob_container:99 - File 'F3554271-Koppert submission for the biotech act consultations.pdf' already exists in container 'brussels-docs'. Skipping upload.
2025-09-02 09:42:31.006 | WARNING  | embeddings:add_file_to_blob_container:99 - File 'F3559517-EBSA Position Paper - OMNIBUS - 20250416 (1).pdf' already exists in container 'brussels-docs'. Skipping upload.
2025-09-02 09:42:31.125 | WARNING  | embeddings:add_file_to_

In [6]:
user_prompt = """
Extract the following fields ONLY from that document; do not invent facts.

Return ONE JSON object with exactly these keys and types:
- "title": string (Do not give back the document name. Instead, find what is the title that appears in the first page of document ?)
- "organisation": array of strings (What is the organizations that wrote this the document)
- "summary": string (summarise the document that you already possess in 4-6 sentences capturing the essentials; suitable for one spreadsheet cell, written in British English)
- "main themes": array of up to 10 short theme phrases (ranked by prominence, written in British English; do not pad with guesses)

Rules:
- Base your answer ONLY on the provided content; do not rely on outside knowledge.
- If a field is not present, set it to null (or [] for arrays).
- Preserve the document's original title (do not translate it); the "summary" and "main_themes" must be in British English.
- Output VALID JSON and NOTHING ELSE (no markdown fences, no prose, no citations like [doc1])."""


In [7]:
pillars_prompt = """Below are concise definitions of five challenges that motivate a new Biotechnology Act:

- VC (Access to Capital): Biotech requires large upfront funding for long R&D, trials, regulation, and scale-up; early bootstrapping helps, but VC/PE/strategic capital typically catalyze growth and Europe faces persistent risk-capital gaps and foreign exits.
- Clusters (Biotech clusters & technology centers): Co-located hubs concentrate talent, infrastructure, and investment, enabling public–private partnerships, sector specialization, shared services (regulatory/funding/market access), and internationalization.
- Skills (Reskilling & upskilling): Biomanufacturing needs deep process expertise (e.g., fermentation, molecular methods) plus cross-cutting data/AI, sustainability, systems thinking, and entrepreneurship; agile VET/HE and lifelong learning are critical.
- AI/data (Use of data & AI): AI (incl. GenAI) turns large scientific/process datasets into gains across discovery, engineering, and operations (e.g., target ID, CRISPR guide design, digital twins), with outcomes dependent on data quality, validation, and governance.
- Dual-use/security (Awareness of misuse risks): Attention to potential dual-use concerns and biosecurity, including risks of misuse such as development of bioweapons, uncontrolled pathogens, or other harmful applications. May appear in the context of regulation, funding decisions, or calls for responsible innovation and safeguards.


TASK: For each challenge, check whether the document makes a substantive reference to it. 
If yes, return a MAXIMUM TWO-SENTENCE summary in BRITISH ENGLISH of how the document addresses that challenge (what it says, proposes, or evidences). If not present, set the value to null.
Return ONE JSON object with EXACTLY these keys (values are either a string ≤2 sentences or null):
{
  "VC": <string or null>,
  "clusters": <string or null>,
  "skills": <string or null>,
  "AI/data": <string or null>,
  "dual_use/security": <string or null>
}

Rules:
- Base your answer ONLY on the provided content; do not rely on outside knowledge.
- Do not invent facts; if uncertain, use null.
- Output VALID JSON and NOTHING ELSE (no markdown fences, no prose, no citations)."""


In [17]:
client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_COGNITIVE_SERVICES_ENDPOINT"],
    api_key=os.environ["AZURE_COGNITIVE_API"],
    api_version="2024-10-21",  # 2024-02-01+ supports data_sources
)

### FIRST TEST BY FILTERING FOR A SINGLE DOCUMENT

In [5]:

def odata_escape(s: str) -> str:
    # OData string literals escape single quotes by doubling them
    return s.replace("'", "''")

In [ ]:
#doc_name = 'F3554271-Koppert submission for the biotech act consultations.pdf'        
doc_name='F3552803-Feedback Paper On the European Biotech Act.pdf'

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[{"role": "user", 
               "content": user_prompt}],
    extra_body={
        "data_sources": [
            {
                "type": "azure_search",
                "parameters": {
                    "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                    "index_name": f"{container_name}_index",
                    "authentication": {
                        "type": "api_key",
                        "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                    },

                    # Map fields so citations show nice titles/paths (optional but handy)
                    "fields_mapping": {
                        "title_field": "title",
                        "filepath_field": "filepath",
                        "url_field": "url",
                        
                    },

                    # B) If you stored the doc name in a custom field
                    "filter": f"title eq '{odata_escape(doc_name)}'",


                    "in_scope": True,        # keep the answer grounded to retrieved docs
                    # "strictness": 4,       # (optional) tighten relevance
                    # "top_n_documents": 5,  # (optional) fewer chunks for stricter focus
                },
            }
        ],
    },
)

print(completion.choices[0].message.content)

In [10]:
def extract_information(doc_name):
    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": f"You already possess the document {doc_name} that contain all necessary information to answer following questions. \n" + user_prompt, }],
        response_format={"type": "json_object"},
        extra_body={
            "data_sources": [
                {
                    "type": "azure_search",
                    "parameters": {
                        "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                        "index_name": f"{container_name}_index",
                        "authentication": {
                            "type": "api_key",
                            "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                        },
                        "fields_mapping": {
                            "title_field": "title",
                            "filepath_field": "filepath",
                            "url_field": "url",
                        },
                        "filter": f"title eq '{odata_escape(doc_name)}'",
                        "in_scope": True,
                        # "strictness": 4,
                        # "top_n_documents": 5,
                    },
                }
            ],
        },
    )
    raw = completion.choices[0].message.content or ""
    try:
        return json.loads(raw)  # now it should be valid JSON
    except json.JSONDecodeError:
        # Fallback: return your schema with nulls so the loop doesn't crash
        return {
            "title": None,
            "organization": [],
            "summary": None,
            "main themes": [],
        }


In [11]:
def extract_pillars(doc_name):
    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": f"You already possess the document {doc_name} that contain all necessary information to answer following questions. \n" + pillars_prompt, }],
        response_format={"type": "json_object"},
        extra_body={
            "data_sources": [
                {
                    "type": "azure_search",
                    "parameters": {
                        "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                        "index_name": f"{container_name}_index",
                        "authentication": {
                            "type": "api_key",
                            "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                        },
                        "fields_mapping": {
                            "title_field": "title",
                            "filepath_field": "filepath",
                            "url_field": "url",
                        },
                        "filter": f"title eq '{odata_escape(doc_name)}'",
                        "in_scope": True,
                        # "strictness": 4,
                        # "top_n_documents": 5,
                    },
                }
            ],
        },
    )
    raw = completion.choices[0].message.content or ""
    try:
        return json.loads(raw)  # now it should be valid JSON
    except json.JSONDecodeError:
        # Fallback: return your schema with nulls so the loop doesn't crash
        return {
            "VC": None,
            "clusters": None,
            "skills": None,
            "AI/data": None,
            "dual_use/security": None
        }


In [78]:
doc_name='F3552803-Feedback Paper On the European Biotech Act.pdf'
extract_pillars(doc_name)

{'VC': 'The document highlights that the EU biotech investment landscape is fragmented, with only 5% of global venture capital compared to 52% in the US, and calls for the establishment of an EU Biotech Investment Facility with blended finance instruments, equity guarantees, and scale-up funding. It also recommends partnering with European stock exchanges to facilitate IPO pathways and reduce reliance on US listings.',
 'clusters': None,
 'skills': "There is a noted mismatch between the labour market and biotech's evolving skills needs, particularly in areas like bioprocess engineering, data science, and regulatory affairs. The document proposes EU-wide talent fellowships and embedding entrepreneurship curricula within Horizon Europe and Erasmus+ to upskill PhD candidates in innovation management and commercialisation.",
 'AI/data': 'The document stresses the importance of access to real-world data, AI-enhanced simulations, and supercomputing resources for biotech SMEs, and recommends 

## LOOP OVER ALL DOCUMENTS

In [104]:

from time import sleep, strftime
from openai import APIConnectionError, BadRequestError
import re

def call_once_with_wait(fn, *args, delay=10, label=None, **kwargs):
    who = f"[{label}]" if label else ""
    print(f"{strftime('%H:%M:%S')} {who} waiting {delay}s before call...", flush=True)
    sleep(delay)
    try:
        out = fn(*args, **kwargs)
        print(f"{strftime('%H:%M:%S')} {who} call OK", flush=True)
        return out
    except BadRequestError as e:
        s = str(e)
        if "Rate limit is exceeded" in s:
            m = re.search(r"Try again in\s+(\d+)\s+seconds?", s, re.I)
            wait_s = int(m.group(1)) + 1 if m else max(5, delay)
            print(f"{strftime('%H:%M:%S')} {who} 429 rate limit → waiting {wait_s}s, then retrying once...", flush=True)
            sleep(wait_s)
            out = fn(*args, **kwargs)
            print(f"{strftime('%H:%M:%S')} {who} retry OK", flush=True)
            return out
        print(f"{strftime('%H:%M:%S')} {who} BadRequestError (not rate-limit): {e}", flush=True)
        raise
    except APIConnectionError as e:
        print(f"{strftime('%H:%M:%S')} {who} connection error → waiting 5s, then retrying once... ({e})", flush=True)
        sleep(5)
        out = fn(*args, **kwargs)
        print(f"{strftime('%H:%M:%S')} {who} retry OK", flush=True)
        return out
    except Exception as e:
        print(f"{strftime('%H:%M:%S')} {who} unexpected error: {e}", flush=True)
        raise


In [ ]:
dir_path = Path(folder_path)  # folder_path already exists
docs = sorted([p.name for p in dir_path.iterdir() if p.is_file()])

results = []
for doc in docs:
    doc_name = doc
    info    = call_once_with_wait(extract_information, doc_name, delay=10, label=f"info:{doc_name}")
    pillars = call_once_with_wait(extract_pillars, doc_name, delay=10, label=f"pillars:{doc_name}")
    results.append({**info, **pillars})

In [105]:
from tqdm.auto import tqdm

dir_path = Path(folder_path)
docs = sorted([p.name for p in dir_path.iterdir() if p.is_file()])

results, errors = [], []
pbar = tqdm(docs, desc="Processing documents", unit="doc", total=len(docs))

for doc in pbar:
    pbar.set_postfix_str(doc)  # show current file name on the bar
    try:
        info    = call_once_with_wait(extract_information, doc, delay=10, label=f"info:{doc}")
        pillars = call_once_with_wait(extract_pillars,     doc, delay=10, label=f"pillars:{doc}")
        results.append({**info, **pillars})
    except Exception as e:
        errors.append((doc, repr(e)))
        pbar.write(f"[ERROR] {doc}: {e}")

# Optional: inspect errors afterwards
# print(errors)


Processing documents:   0%|          | 0/149 [00:00<?, ?doc/s, F3552803-Feedback Paper On the European Biotech Act.pdf]

18:21:55 [info:F3552803-Feedback Paper On the European Biotech Act.pdf] waiting 10s before call...


18:22:12 [info:F3552803-Feedback Paper On the European Biotech Act.pdf] call OK
18:22:12 [pillars:F3552803-Feedback Paper On the European Biotech Act.pdf] waiting 10s before call...
18:22:29 [pillars:F3552803-Feedback Paper On the European Biotech Act.pdf] call OK


Processing documents:   1%|          | 1/149 [00:34<1:25:39, 34.73s/doc, F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] 

18:22:29 [info:F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] waiting 10s before call...
18:22:44 [info:F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] call OK
18:22:44 [pillars:F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] waiting 10s before call...
18:22:58 [pillars:F3553888-cpme.2025-092.FINAL.Statement.Biotech.Act.pdf] call OK


...

Processing documents:  99%|█████████▊| 147/149 [1:20:11<01:03, 31.73s/doc, F3565532-BiotechAct_VPHi.pdf]                    

19:42:06 [info:F3565532-BiotechAct_VPHi.pdf] waiting 10s before call...
19:42:23 [info:F3565532-BiotechAct_VPHi.pdf] call OK
19:42:23 [pillars:F3565532-BiotechAct_VPHi.pdf] waiting 10s before call...
19:42:37 [pillars:F3565532-BiotechAct_VPHi.pdf] call OK


Processing documents:  99%|█████████▉| 148/149 [1:20:42<00:31, 31.58s/doc, F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx]

19:42:37 [info:F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx] waiting 10s before call...
19:42:54 [info:F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx] call OK
19:42:54 [pillars:F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx] waiting 10s before call...
19:43:08 [pillars:F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx] call OK


Processing documents: 100%|██████████| 149/149 [1:21:13<00:00, 32.71s/doc, F3582124-Q12_cfe_biotech_act_Fondazione Telethon.docx]


In [106]:
# --- make a DataFrame with your exact columns ---
def _tolist(x):
    if isinstance(x, list):
        return x
    return [x] if x else []

def _tostr_or_none(x):
    if x is None:
        return None
    return x if isinstance(x, str) else str(x)

rows = []
for r in results:
    rows.append({
        "title": r.get("title"),
        "organisation": _tolist(r.get("organisation")),     # keep as list
        "page_number": r.get("page_number"),
        "summary": r.get("summary"),
        "main themes": _tolist(r.get("main themes")),       # keep as list
        "VC": _tostr_or_none(r.get("VC")),                  # strings or None
        "clusters": _tostr_or_none(r.get("clusters")),
        "skills": _tostr_or_none(r.get("skills")),
        "AI/data": _tostr_or_none(r.get("AI/data")),
        "dual_use/security":_tostr_or_none(r.get("dual_use/security"))
    })

df = pd.DataFrame(
    rows,
    columns=["title", "organisation", "page_number", "summary", "main themes",
             "VC", "clusters", "skills", "AI/data","dual_use/security"]
)

In [107]:
df

,title,organisation,page_number,summary,main themes,VC,clusters,skills,AI/data,dual_use/security
0,Feedback on the European Biotech Act,[RareGen Youth Network],None,This document provides feedback on the Europea...,"[Regulatory fragmentation and bottlenecks, Mar...",The document highlights persistent gaps in ris...,The document discusses the importance of biote...,A shortage of skilled biotech professionals is...,The document stresses the growing importance o...,The document raises concerns about dual-use ri...
1,Statement on the European Commission call for ...,[Standing Committee of European Doctors (CPME)],None,The document is a statement from the Standing ...,"[Biotech Act consultation, Transparency in the...",The document urges the European Commission to ...,None,None,None,None
2,Koppert’s Contribution to the EU Biotech Act C...,[Koppert],None,Koppert's submission to the EU Biotech Act con...,"[Biocontrol inclusion in EU Biotech Act, Regul...",The document highlights the need for the EU to...,It advocates for the creation of EU innovation...,None,The document proposes supporting public–privat...,None
3,Maturing Biological Risk Management for EU com...,[European Biosafety Association (EBSA)],None,This position paper by the European Biosafety ...,"[Regulatory simplification in life sciences, B...",None,None,None,None,The document highlights the need for a unified...
4,PhageEU proposals for amendments to Pharmaceut...,[PhageEU Coalition],None,This document proposes amendments to the EU Ph...,[Integration of phage therapies into EU pharma...,The document highlights that explicit inclusio...,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...
144,An agroecological Europe in 2050: multifunctio...,"[IDDRI, AScA]",None,This document presents the TYFA (Ten Years for...,"[Agroecological transition, Biodiversity resto...",None,None,None,None,None
145,Feedback on the Call for Evidence on the Biote...,[GreenLight Biosciences],None,This document provides GreenLight Biosciences'...,"[Regulatory simplification for biocontrol, Fun...",The document highlights significant funding ch...,The document advocates for the creation of res...,It notes that while Europe has a strong talent...,None,None
146,Organisational Background,[Bio Base Europe Pilot Plant],None,The document outlines the role of Bio Base Eur...,[Open access pilot and demonstration facilitie...,The document highlights that European support ...,None,The document stresses the importance of ensuri...,The document advocates for modernising pilot a...,None
147,Virtual Physiological Human Institute for Inte...,[Virtual Physiological Human Institute IVZW - ...,None,This document presents the Virtual Physiologic...,"[Digital technologies in biotechnology, Artifi...",None,None,None,The document highlights the strategic applicat...,None


### Using PdfReader for page count

In [108]:
paths = sorted([p for p in Path(folder_path).iterdir() if p.is_file()])[:len(df)]

# Optional: DOCX/DOC via Microsoft Word (Windows only)
try:
    WORD_AVAILABLE = True
    word = win32.gencache.EnsureDispatch("Word.Application")
    word.Visible = False
except Exception:
    WORD_AVAILABLE = False
    word = None

def count_pages(p: Path):
    suf = p.suffix.lower()
    if suf == ".pdf":
        try:
            with open(p, "rb") as f:
                r = PdfReader(f)
                if getattr(r, "is_encrypted", False):
                    try: r.decrypt("")
                    except Exception: return np.nan
                return len(r.pages)
        except Exception:
            return np.nan
    if suf in (".docx", ".doc") and WORD_AVAILABLE:
        try:
            doc = word.Documents.Open(str(p))
            pages = int(doc.ComputeStatistics(2))  # 2 = wdStatisticPages
            doc.Close(False)
            return pages
        except Exception:
            return np.nan
    return np.nan

page_counts = pd.Series([count_pages(p) for p in paths], index=df.index).astype("Int64")

if word: 
    try: word.Quit()
    except Exception: pass

if "page_number" in df.columns:
    df = df.drop(columns=["page_number"])
df.insert(2, "page_number", page_counts)


df.insert(0, "file_name", pd.Series(docs[:len(df)], index=df.index))


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong point

In [110]:
output_folder = Path(folder_path).parent / "results"
xlsx_path = Path(output_folder) / "RAG_all_results.xlsx"

df.to_excel(xlsx_path, index=False)
print(f"Saved: {xlsx_path}")

Saved: C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\results\RAG_all_results.xlsx


In [111]:
df[-30:]

,file_name,title,organisation,page_number,summary,main themes,VC,clusters,skills,AI/data,dual_use/security
119,F3565383-New Harvest NL EUBiotechAct.pdf,In support of The EU Biotech Act by Stichting ...,[Stichting New Harvest Netherlands],4,This document presents recommendations from St...,"[Cellular agriculture and novel foods, Regulat...",The document recommends that the EU create its...,The document proposes the creation of open acc...,The document calls for boosting research and a...,The document emphasises the importance of shar...,None
120,F3565386-20250611-EBIC-BiotechAct-Submission.pdf,EBIC’s Contribution to the European Commission...,[European Biostimulants Industry Council (EBIC)],4,This document presents EBIC's feedback on the ...,"[Regulatory coherence for biostimulants, Marke...",The document highlights that regulatory fragme...,None,None,None,None
121,F3565387-BIC contribution to Biotech Act call ...,BIC contribution to the call for evidence on E...,[BIC],2,The document provides BIC's input to the EU Bi...,"[Expanding the scope of the EU Biotech Act, St...",The document highlights the persistent challen...,None,The document stresses the necessity of investi...,None,None
122,F3565388-2025 Biotech Act Call for evidence - ...,IFF case studies for Call for Evidence on Biot...,[IFF],3,This document presents IFF's contributions to ...,"[Regulatory classification of biomaterials, Su...",The document highlights that current EU regula...,None,None,None,None
123,F3565395-Lallemand Call for evidence EU Biotec...,LALLEMAND’S COMMENTS TO THE EUROPEAN COMMISSIO...,[Lallemand],4,The document presents Lallemand's response to ...,[Broad and ambitious scope for the Biotech Act...,None,None,The document highlights that Europe's biotechn...,None,None
124,F3565399-Response from the European Respirator...,Response from the European Respiratory Society...,"[European Respiratory Society (ERS), European ...",3,The document is a response from the European R...,[Regulatory simplification with safety safegua...,None,None,The document highlights the need to strengthen...,The document supports the development of digit...,None
125,F3565405-20250611_Pharma Deutschland_Call for ...,POSITION PAPER of Pharma Deutschland e.V. on t...,[Pharma Deutschland e.V.],4,This position paper by Pharma Deutschland e.V....,"[Standardisation of regulatory requirements, R...",The document highlights the need to enhance in...,None,None,"The document recommends establishing agile, se...",None
126,F3565413-20250611IPAEUROPE Biotech Act Call fo...,INTERNATIONAL PROBIOTICS ASSOCIATION EUROPE,[International Probiotics Association Europe],3,The document highlights the transformative pot...,"[Fermentation technology and biosolutions, EU ...",None,None,None,The document proposes integrating data and AI ...,None
127,F3565422-EU Biotech Act_CSL Response.pdf,CSL Response to Call for Evidence for the EU B...,"[CSL, CSL Behring, CSL Vifor, CSL Seqirus]",4,This document is CSL's response to the Europea...,"[Regulatory harmonisation and simplification, ...",The document suggests considering the implemen...,The document recommends prioritising the devel...,The document calls for increased EU funding an...,The document advocates for advancing risk-adap...,None
128,F3565426-AmchamEU_aeu_call_for_evidence_respon...,Response to the Commission’s call for evidence...,[American Chamber of Commerce to the European ...,3,This document is AmCham EU's response to the E...,"[Regulatory coherence in biotech, Global compe...",The document highlights the need for EU-level ...,The document notes that European biotech clust...,None,The document states that AI and advanced compu...,The document recommends that the Biotech Act s...


### Treating the Missings

In [36]:
output_folder = Path(folder_path).parent / "results"
xlsx_path = Path(output_folder) / "RAG_all_results.xlsx"
df = pd.read_excel(xlsx_path)

In [38]:
       
doc_name='F3565290-UniSR_European Biotech Act_Extended.pdf'
#doc_name='F3564682-Reco EU biotech EWPA EWPM 10.06.2025.pdf'

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[{"role": "user", 
               "content": user_prompt}],
    extra_body={
        "data_sources": [
            {
                "type": "azure_search",
                "parameters": {
                    "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                    "index_name": f"{container_name}_index",
                    "authentication": {
                        "type": "api_key",
                        "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                    },

                    # Map fields so citations show nice titles/paths (optional but handy)
                    "fields_mapping": {
                        "title_field": "title",
                        "filepath_field": "filepath",
                        "url_field": "url",
                        
                    },

                    # B) If you stored the doc name in a custom field
                    "filter": f"title eq '{odata_escape(doc_name)}'",


                    "in_scope": True,        # keep the answer grounded to retrieved docs
                    # "strictness": 4,       # (optional) tighten relevance
                    # "top_n_documents": 5,  # (optional) fewer chunks for stricter focus
                },
            }
        ],
    },
)

print(completion.choices[0].message.content)

{
  "title": "Contribution to the Consultation on the Future European Biotech Act",
  "organisation": ["Università Vita-Salute San Raffaele"],
  "summary": "The document is a contribution from Università Vita-Salute San Raffaele to the European Commission's consultation on the future European Biotech Act. It welcomes the initiative and provides recommendations to strengthen Europe's position in biotechnology. Key suggestions include enhanced support for academic centres, streamlined regulatory processes, and the creation of a 'Single Biotech Permit' for EU-wide operations. The document also advocates for a dedicated translational track for biotech projects and harmonised tax incentives for private investment. Additionally, it highlights the importance of secure and efficient access to anonymised patient data. The overall aim is to reduce barriers and accelerate innovation in the European biotech sector.",
  "main themes": [
    "Support for academic biotech research",
    "Regulatory s

In [39]:

extract_pillars(doc_name)

{'VC': 'The document proposes an EU framework for tax relief to incentivise high-risk investment in biotech, including tax credits, capital gains tax deferrals, and favourable treatment of convertible instruments. It argues that harmonising such incentives would create a more predictable and attractive landscape for both European and international investors.',
 'clusters': 'The document recommends dedicated support for academic centres to play a leading role in future biotech clusters and centres of excellence across the EU. It also suggests the creation of themed pan-European biotech accelerators to systematically transform research outputs into companies with access to capital, talent, and infrastructure.',
 'skills': None,
 'AI/data': 'The document highlights the importance of real-world data and AI in accelerating the development of innovative therapies, calling for dedicated support for AI technologies that leverage such data. It emphasises the need for collaboration between biote

In [40]:
main_themes = [
    "Support for academic biotech research",
    "Regulatory streamlining",
    "Single Biotech Permit",
    "Translational track for biotech projects",
    "Tax incentives for biotech investment",
    "Data access and security",
    "Innovation acceleration",
    "EU-wide harmonisation"
  ]

print(main_themes)

['Support for academic biotech research', 'Regulatory streamlining', 'Single Biotech Permit', 'Translational track for biotech projects', 'Tax incentives for biotech investment', 'Data access and security', 'Innovation acceleration', 'EU-wide harmonisation']
